# Advanced Reasoning — Self-Consistency, Decomposition, and Tree of Thought

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q boto3 openai tiktoken anthropic matplotlib

## How LLM calls work in this notebook

Every live API call goes through `LLMRouter` from `garage_helper`:

1. **Model resolution** — the router maps your model to the right provider (`anthropic`, `openai`, `azure_openai`, `bedrock_claude`, `gemini`, etc.) automatically.
2. **Self-consistency** — runs `N` independent `router.generate()` calls at `temperature=0.7`, each returning a separate chain. The provider instance is cached so the API client is created once and reused for all N calls.
3. **Least-to-most** — makes a sequence of `router.generate()` calls where each call's answer is embedded in the next prompt. The router knows nothing about this chaining — it is plain Python driving sequential calls.
4. **Tree of Thought** — interleaves generation and scoring calls. Each `generate_thoughts` and `score_thought` call is a separate dispatch through the active provider.
5. **Token tracking** — `router.generate_response()` returns `.input_tokens + .output_tokens` per call so you can sum the total cost of each multi-call technique.

These notebooks have been tested with **Claude** (via Anthropic direct API and AWS Bedrock) and **GPT** models (via Azure OpenAI and direct OpenAI). Set `verbose=True` to see each individual API call logged in cell output.

In [ ]:
import sys
import os
import re
from collections import Counter
sys.path.insert(0, os.path.abspath("../.."))

from garage_helper import setup_llm, LLMRouter

# Contributors: add DEFAULT_LLM_MODEL (and any provider credentials) to a .env at the repo root.
# Learners: setup_llm() will run an interactive wizard to pick a provider and enter credentials.
MODEL  = setup_llm()
router = LLMRouter(default_model=MODEL, verbose=False)

## When a single chain of thought isn't reliable enough

In the previous notebook we saw how Chain-of-Thought prompting dramatically improves accuracy on multi-step problems by externalising intermediate reasoning into the context window. For many tasks that is the end of the story — CoT gets you there.

But there is a class of problems where even a well-formed chain of thought fails often enough to matter. The model picks a reasoning path early, commits to it, and follows it to a wrong conclusion with complete apparent confidence. The chain looks valid on the surface; the answer is wrong. You read the steps and can't immediately spot the error. This is not a formatting problem or a prompt clarity problem — it is a search problem. A single chain only explores one path through a space that might have dozens.

This notebook covers three techniques that go beyond a single chain:

**Self-consistency** runs many chains independently and takes a majority vote — trading parallelism for reliability. **Least-to-most prompting** breaks a hard problem into ordered subproblems and solves them sequentially, feeding each answer forward — trading depth for tractability. **Tree of Thought** explicitly branches the reasoning at key decision points, evaluates each branch, and prunes dead ends before committing — trading cost for thoroughness.

The theme running through all three: more reasoning buys more accuracy, and it always costs more tokens. Knowing which technique fits which problem — and when to stop escalating — is the skill this notebook builds.

## Concept 1 — Self-consistency: sample many chains, take a vote

Self-consistency is the simplest upgrade from single-chain CoT. Instead of generating one reasoning chain and trusting it, you generate N chains independently (each one slightly different because of sampling temperature) and take the majority answer.

**The analogy:** imagine you are at a hospital with an ambiguous X-ray. You could get one radiologist's opinion and trust it. Or you could send the scan to five independent radiologists, each writing their own interpretation, then go with whatever four of the five agree on. Each radiologist might make a different mistake — but genuine errors rarely cluster on the same wrong answer. The majority vote surfaces the answer that survives independent scrutiny.

This was introduced in Wang et al. (2022) and showed that majority voting over 40 chains improved accuracy on grade-school math benchmarks by 17 percentage points over greedy single-chain CoT. The intuition is that correct reasoning paths tend to converge on the same answer, while errors scatter across many wrong ones. You don't need 40 chains in practice — 5 to 10 is usually enough to see the benefit.

The cost is linear: N chains costs exactly N times a single CoT call. That is the tradeoff — you are buying reliability with parallelism.

In [ ]:
problem = """
A factory produces widgets in batches. Each batch requires 3 hours of machine time
and yields 144 widgets. The machine runs for 11 hours a day. Each widget sells for
£0.85. Fixed daily overhead is £120. How much daily profit does the factory make?
""".strip()

system = "You are a precise problem solver. Think step by step, then end with 'Answer: £<number>'."

N_SAMPLES = 7
answers   = []

print(f"Running {N_SAMPLES} independent reasoning chains at temperature=0.7...")
print()

for i in range(N_SAMPLES):
    text = router.generate(problem, model=MODEL, system=system, max_tokens=400, temperature=0.7)
    match  = re.search(r'Answer:\s*[£$]?\s*([\d,\.]+)', text, re.IGNORECASE)
    answer = match.group(1).replace(',', '') if match else "unknown"
    answers.append(answer)
    print(f"  Chain {i+1}: Answer = £{answer}")

vote_counts     = Counter(answers)
majority_answer, majority_count = vote_counts.most_common(1)[0]

print()
print(f"Vote tally: {dict(vote_counts)}")
print(f"Majority answer: £{majority_answer}  ({majority_count}/{N_SAMPLES} chains agreed)")
print()
print("Expected: 3 complete batches in 11h → 432 widgets → £367.20 revenue → £247.20 profit")
print(f"Self-consistency cost: {N_SAMPLES}× a single CoT call — but the majority vote is far harder to fool.")

## Concept 2 — Least-to-most prompting: solve subproblems in order

Self-consistency attacks unreliability by sampling many independent chains. Least-to-most attacks a different failure mode: problems where the question itself is too hard to tackle in one shot, but becomes tractable if you break it into ordered pieces.

**The analogy:** nobody builds a skyscraper by starting at the top floor. You pour the foundation, then the ground floor, then the next — each floor becomes the literal platform for the one above it. Trying to skip straight to the top floor and reason backwards is not engineering, it's guessing. Least-to-most prompting is the foundation-first construction plan: identify the simplest subproblem the full problem depends on, solve it, then use that answer as the foundation for the next subproblem up the chain.

Proposed by Zhou et al. (2022), least-to-most is particularly powerful on tasks where intermediate results are reused — programming problems, multi-hop reasoning, anything where step N requires the verified answer from step N-1. Unlike CoT which generates all steps in one pass, least-to-most can verify each subproblem before proceeding, and the earlier answers appear in context as grounded facts rather than provisional guesses.

In [ ]:
hard_problem = """
A courier company charges by zone. Zone A deliveries cost £3.50 each, Zone B cost £5.20 each.
On Monday: 24 Zone A and 15 Zone B deliveries.
On Tuesday: 18 Zone A and 22 Zone B deliveries.
The company pays drivers £0.40 per delivery plus a flat £45 daily wage each.
There are 3 drivers. What is the company's two-day net revenue after driver costs?
""".strip()

system = "You are a precise accountant. Be concise and exact."

def ask(prompt: str, max_tokens: int = 120) -> str:
    return router.generate(prompt, model=MODEL, system=system, max_tokens=max_tokens)

# --- Single-shot CoT ---
single_shot = ask(hard_problem + " Think step by step.", max_tokens=350)

# --- Least-to-most: four grounded layers ---
a1 = ask(
    "Monday: 24 Zone A and 15 Zone B deliveries. Tuesday: 18 Zone A and 22 Zone B deliveries. "
    "What is the total number of deliveries each day, and overall?"
)
a2 = ask(
    "Zone A costs £3.50 each, Zone B costs £5.20 each. "
    "Monday: 24 Zone A, 15 Zone B. Tuesday: 18 Zone A, 22 Zone B. "
    "What is the total revenue across both days?"
)
a3 = ask(
    "There are 3 drivers, each paid £45/day flat plus £0.40 per delivery. "
    "Monday had 39 deliveries, Tuesday had 40. "
    "What is the total driver cost across both days?"
)
a4 = ask(
    f"Revenue across two days: {a2}. "
    f"Driver costs across two days: {a3}. "
    "What is the net revenue after driver costs?"
)

print("=" * 60)
print("SINGLE-SHOT CoT:")
print(single_shot)

print()
print("=" * 60)
print("LEAST-TO-MOST — four grounded layers:")
print(f"  Layer 1 — delivery counts : {a1}")
print(f"  Layer 2 — total revenue   : {a2}")
print(f"  Layer 3 — driver costs    : {a3}")
print(f"  Layer 4 — net revenue     : {a4}")

print()
print("Expected net revenue: £37.80")
print("Least-to-most verifies each layer before the next builds on it — errors can't compound silently.")

## Concept 3 — Tree of Thought: explore branches, prune dead ends

Self-consistency and least-to-most still commit to a linear reasoning structure — one chain at a time, one subproblem at a time. Tree of Thought (ToT) goes further: it explicitly generates *multiple candidate reasoning paths* at each decision point, evaluates which ones look promising, and abandons the ones that don't — exactly like a chess engine evaluating positions several moves ahead.

**The analogy:** you are trying to escape a hedge maze. Self-consistency sends five people in independently and asks which exit they found most. Least-to-most sends one person who maps one corridor at a time. Tree of Thought sends one person who, at every fork, marks both paths, walks each one a few steps, evaluates which looks more open, then commits to the better one and abandons the dead end. It is a search strategy, not just a reasoning strategy.

Proposed by Yao et al. (2023), ToT works by interleaving generation and evaluation. The model generates a set of "thoughts" (partial reasoning steps), scores each for plausibility, selects the best candidates, and expands them. This is most valuable on problems where early mistakes are hard to recover from and where the search space has genuine branching structure — combinatorial puzzles, code design decisions, planning tasks with constraints.

The cost scales with breadth × depth: if you explore B branches at each of D levels, you make B^D reasoning calls. That is expensive. ToT is the most powerful technique in this notebook and the most expensive one to run.

In [ ]:
NUMBERS = [3, 8, 3, 8]
TARGET  = 24

system = "You are a mathematical puzzle solver. Be concise and precise."

def generate_thoughts(context: str, n: int = 3) -> list:
    prompt = (
        f"{context}\n\n"
        f"Propose {n} distinct next arithmetic steps (each using exactly one operation "
        f"on two of the remaining numbers). Number each proposal. Be concise."
    )
    text  = router.generate(prompt, model=MODEL, system=system, max_tokens=250)
    lines = [l.strip() for l in text.split('\n') if l.strip() and l.strip()[0].isdigit()]
    return lines[:n] if lines else [text]

def score_thought(thought: str, remaining_context: str) -> float:
    prompt = (
        f"Problem: reach {TARGET} using {NUMBERS} with +, -, *, / (each number once).\n"
        f"Current path: {remaining_context}\n"
        f"Next step considered: {thought}\n\n"
        f"Rate how promising this step is (0=dead end, 10=very likely to reach {TARGET}). "
        f"Reply with just a single integer 0–10."
    )
    text = router.generate(prompt, model=MODEL, system=system, max_tokens=5)
    try:
        return float(re.search(r'\d+', text).group())
    except Exception:
        return 0.0

root_context = f"Goal: reach {TARGET} using the numbers {NUMBERS} with +, -, *, / (each number once)."

print(f"Problem: reach {TARGET} using {NUMBERS}")
print("=" * 60)

print("\n[Level 1 — generating candidate first steps]")
level1_thoughts = generate_thoughts(root_context, n=3)
level1_scored   = []
for thought in level1_thoughts:
    score = score_thought(thought, root_context)
    level1_scored.append((thought, score))
    print(f"  Step: {thought[:70]}")
    print(f"  Score: {score}/10")
    print()

level1_scored.sort(key=lambda x: x[1], reverse=True)
top2 = level1_scored[:2]
print(f"  → Pruned: keeping top 2 branches (scores: {top2[0][1]}, {top2[1][1]})")

print("\n[Level 2 — expanding top branches]")
final_candidates = []
for branch_thought, branch_score in top2:
    expanded_context = root_context + f"\nStep taken: {branch_thought}"
    next_thoughts    = generate_thoughts(expanded_context, n=2)
    for nt in next_thoughts:
        score = score_thought(nt, expanded_context)
        final_candidates.append((branch_thought, nt, score))
        print(f"  Branch: {branch_thought[:50]}...")
        print(f"    Next: {nt[:60]}")
        print(f"    Score: {score}/10")
        print()

best = max(final_candidates, key=lambda x: x[2])
print("=" * 60)
print(f"Best path found:")
print(f"  Step 1: {best[0]}")
print(f"  Step 2: {best[1]}")
print(f"  Final score: {best[2]}/10")
print()
print(f"Known solution: 8 / (3 - 8/3) = 24")
print("ToT explored multiple branches and pruned lower-scoring paths before committing.")

## Concept 4 — The token cost ladder: paying for reasoning

Every technique in this section improves accuracy on hard problems. None of them is free. It is worth mapping out exactly what each one costs relative to a direct answer, so you can choose deliberately rather than reaching for the most powerful technique by default.

**The analogy:** think of reasoning techniques as modes of transport. Walking (direct answer) is free and fine for distances under 500 metres. A bicycle (CoT) is faster and reliable for a few kilometres. A taxi (self-consistency) is comfortable and predictable but you pay per trip multiplied by number of passengers. A private jet (Tree of Thought) gets you there fastest and can reroute mid-flight, but the hourly rate is a different order of magnitude. Nobody hires a private jet to get to the corner shop.

The right question is not "which technique is most accurate?" It is: "given the hardness of this problem and the cost of being wrong, what is the cheapest technique that clears the accuracy bar I need?"

In [ ]:
problem = """
A conference has 4 session tracks running in parallel. Each track has 6 sessions.
Each session lasts 45 minutes with 15-minute breaks between sessions.
The conference runs 8 hours each day for 2 days.
How many total sessions can actually be completed across the full conference?
""".strip()

system = "You are a precise planner."
total_tokens = {}

# Direct
r = router.generate_response(problem, model=MODEL, system=system, max_tokens=80)
total_tokens["Direct"] = r.input_tokens + r.output_tokens
direct_answer = r.text.strip()

# Single CoT
r = router.generate_response(
    problem + " Think step by step.", model=MODEL, system=system, max_tokens=350
)
total_tokens["CoT (1 chain)"] = r.input_tokens + r.output_tokens
cot_answer = r.text.strip()

# Self-consistency (5 chains)
sc_tok, sc_answers = 0, []
for _ in range(5):
    r = router.generate_response(
        problem + " Think step by step. End with 'Answer: <number>'",
        model=MODEL, system=system, max_tokens=350, temperature=0.7
    )
    sc_tok += r.input_tokens + r.output_tokens
    m = re.search(r'Answer:\s*(\d+)', r.text)
    sc_answers.append(m.group(1) if m else "?")
sc_majority = Counter(sc_answers).most_common(1)[0][0]
total_tokens["Self-consistency (5×)"] = sc_tok

# Least-to-most (2 layers)
ltm_tok = 0
r1 = router.generate_response(
    "Each session is 45 minutes with a 15-minute break after it. "
    "How many complete sessions fit in an 8-hour (480-minute) day? Give just the number.",
    model=MODEL, system=system, max_tokens=80
)
sessions_per_day = r1.text.strip()
ltm_tok += r1.input_tokens + r1.output_tokens

r2 = router.generate_response(
    f"A conference has 4 parallel tracks. Each track can run {sessions_per_day} sessions per day. "
    "The conference lasts 2 days. How many total sessions is that across all tracks? Give just the number.",
    model=MODEL, system=system, max_tokens=80
)
ltm_total = r2.text.strip()
ltm_tok += r2.input_tokens + r2.output_tokens
total_tokens["Least-to-most (2 layers)"] = ltm_tok

base = total_tokens["Direct"]
print(f"Expected answer: 64 sessions (8 slots/track/day × 4 tracks × 2 days)")
print()
print(f"{'Technique':<26}  {'Tokens':>8}  {'vs Direct':>10}  {'Answer'}")
print("-" * 70)
answers_map = {
    "Direct":                  direct_answer.split('\n')[0][:30],
    "CoT (1 chain)":           cot_answer.split('\n')[-1][:30],
    "Self-consistency (5×)":   f"majority = {sc_majority} (votes: {dict(Counter(sc_answers))})",
    "Least-to-most (2 layers)": ltm_total[:30],
}
for technique, tok in total_tokens.items():
    multiplier = tok / base
    print(f"{technique:<26}  {tok:>8}  {multiplier:>9.1f}×  {answers_map[technique]}")

print()
print("Token cost grows with reasoning sophistication. The cost premium only pays off when techniques diverge.")

## Putting it together — choosing the right technique

Five techniques, one framework. Here is how to decide which to reach for.

In [ ]:
ROUTER_SYSTEM = """
You classify problems by the reasoning technique most appropriate for them.
Reply with exactly one of: DIRECT, COT, SELF_CONSISTENCY, LEAST_TO_MOST, TREE_OF_THOUGHT

Rules:
  DIRECT           — factual lookup, single-step, no chaining needed
  COT              — multi-step but linear; one correct path exists and is findable in one pass
  SELF_CONSISTENCY — multi-step and numerically sensitive; small errors compound; single chain unreliable
  LEAST_TO_MOST    — problem has clear prerequisite layers; later steps require verified earlier answers
  TREE_OF_THOUGHT  — solution space has genuine branches; wrong early choices are hard to recover from
""".strip()

def route(problem: str) -> str:
    return router.generate(problem, model=MODEL, system=ROUTER_SYSTEM, max_tokens=10).strip()

cost_tier = {
    "DIRECT":           "1×",
    "COT":              "3–5×",
    "SELF_CONSISTENCY": "15–25×",
    "LEAST_TO_MOST":    "8–15×",
    "TREE_OF_THOUGHT":  "20–100×",
}

test_problems = [
    ("What year was the Python programming language first released?",
     "DIRECT"),
    ("A cyclist rides 18km at 12km/h then 24km at 16km/h. What is their average speed for the whole journey?",
     "COT"),
    ("A portfolio has 12 stocks. Each stock returns between -8% and +22% this year. "
     "Given complex correlation rules between sectors, what is the expected portfolio return?",
     "SELF_CONSISTENCY"),
    ("Calculate the total cost of a multi-stage manufacturing process where each stage's "
     "output quantity and unit cost feeds into the next stage's input.",
     "LEAST_TO_MOST"),
    ("Design an optimal seating arrangement for a dinner party of 12 where 6 pairs of guests "
     "must not sit adjacent and 3 pairs must sit together.",
     "TREE_OF_THOUGHT"),
]

print(f"{'Problem (truncated)':<55}  {'Routed to':<20}  {'Cost'}  {'Expected?'}")
print("-" * 105)

for problem, expected in test_problems:
    routed = route(problem)
    cost   = cost_tier.get(routed, "?")
    match  = "✓" if routed == expected else f"✗ (expected {expected})"
    print(f"{problem[:55]:<55}  {routed:<20}  {cost:<7}  {match}")

print()
print("Routing lets you apply the minimum necessary technique automatically.")
print("The goal is not to always use the most powerful method — it's to use the cheapest one that works.")

## Visualising the accuracy–cost tradeoff across all five techniques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Illustrative data based on published benchmarks
# (GSM8K, MATH, BIG-Bench Hard — values representative, not exact per paper)

techniques = [
    "Direct",
    "CoT",
    "Least-to-most",
    "Self-consistency\n(N=5)",
    "Tree of Thought",
]

# Accuracy on hard multi-step reasoning tasks
accuracy = [0.42, 0.67, 0.75, 0.82, 0.88]

# Relative token cost vs Direct (log scale makes sense here)
token_cost = [1, 4, 10, 20, 60]

colours = ["steelblue", "seagreen", "#d4a843", "tomato", "#8e44ad"]
sizes   = [120, 140, 160, 180, 200]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter — cost vs accuracy (the core tradeoff)
for i, (name, acc, cost, col, sz) in enumerate(zip(techniques, accuracy, token_cost, colours, sizes)):
    ax1.scatter(cost, acc, color=col, s=sz, zorder=3, edgecolors='white', linewidths=1.5)
    label_x_offset = [3, 3, 3, -18, 3]
    label_y_offset = [0.01, 0.01, -0.03, 0.01, 0.01]
    ax1.annotate(
        name.replace('\n', ' '),
        (cost, acc),
        xytext=(cost + label_x_offset[i], acc + label_y_offset[i]),
        fontsize=8.5,
    )

# Frontier curve
ax1.plot(token_cost, accuracy, color='grey', linewidth=1, linestyle='--', alpha=0.5)

ax1.set_xscale('log')
ax1.set_xlabel("Relative token cost (log scale, Direct = 1×)")
ax1.set_ylabel("Accuracy on hard multi-step problems (illustrative)")
ax1.set_title("The accuracy–cost frontier")
ax1.set_ylim(0.3, 1.0)
ax1.axhline(0.8, color='grey', linewidth=0.7, linestyle=':', alpha=0.5)
ax1.text(1.1, 0.81, "80% target", fontsize=8, color='grey')
ax1.grid(True, alpha=0.2)

# Right: stacked bar showing where tokens go for each technique
categories = ["Direct", "CoT", "Least-to-\nmost", "Self-cons.\n(N=5)", "ToT"]
base_call   = [1,  1,  3,  5, 10]   # number of API calls
tokens_each = [1,  4,  4,  4,  6]   # tokens per call (relative)

x = np.arange(len(categories))
total_bars  = [b * t for b, t in zip(base_call, tokens_each)]

bar_colours = ["steelblue", "seagreen", "#d4a843", "tomato", "#8e44ad"]
ax2.bar(x, total_bars, color=bar_colours, alpha=0.85)

for xi, val in zip(x, total_bars):
    ax2.text(xi, val + 0.5, f"{val}×", ha='center', fontsize=9)

ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=9)
ax2.set_ylabel("Relative total token cost (Direct = 1×)")
ax2.set_title("Total token cost by technique")
ax2.set_ylim(0, 75)

plt.tight_layout()
plt.show()

print("Left chart: every technique sits on a cost-accuracy frontier — more cost, more accuracy.")
print("Left chart: the gap between CoT and ToT is large in cost but only ~20pp in accuracy.")
print("Right chart: ToT costs 60× a direct answer. Self-consistency costs 20×.")
print("Key insight: pick the cheapest technique that clears your accuracy requirement — not the most powerful one.")

## Key takeaways

- **Self-consistency** runs N independent reasoning chains and takes the majority vote. Correct paths converge; errors scatter. It is the easiest upgrade from single-chain CoT and scales linearly in cost — N chains costs exactly N×.
- **Least-to-most prompting** decomposes a hard problem into ordered prerequisite layers and solves them sequentially, feeding each verified answer forward. It prevents errors from compounding silently across dependent steps.
- **Tree of Thought** explicitly branches at decision points, scores each branch, and prunes dead ends before committing. It is the most powerful technique here and the most expensive — use it when early wrong choices are catastrophic and the search space has genuine branching structure.
- **Every technique lives on a cost-accuracy frontier** — more reasoning always costs more tokens. There is no free lunch. The right question is: what is the cheapest technique that clears the accuracy bar I need?
- **A routing layer pays for itself** on production systems where request difficulty varies: cheap requests stay cheap, hard requests get the technique they need, and you don't pay ToT prices for factual lookups.
- **Self-consistency is the pragmatic default** for numerically sensitive problems where you can't afford a wrong answer but also can't afford full ToT complexity. Five chains is usually enough.
- **These techniques compose**: least-to-most can use self-consistency within each layer; ToT can use CoT at each node. The techniques are building blocks, not mutually exclusive choices.

---

Next up: **ReAct and Reflection** — what happens when the model needs to reason *and* act in the world: tool use, observation loops, and how agents check their own work before committing to a final answer.